In [1]:
from dotenv import load_dotenv

load_dotenv()

True

<div style="font: bold 40px sans-serif;">LangChain Runtime 运行时机制</div>

<br/>
LangChain的Runtime机制是理解Agent内部运行状态的关键。它包括三个核心概念：


| 概念 | 说明                      | 生命周期 |
|-----|-------------------------|---------|
| **State** | 短期记忆，存储Agent当前对话信息、任务状态 | 单次请求 |
| **Store** | 长期记忆，包含用户偏好、失败经验等       | 跨会话 |
| **Context** | 运行时上下文，传递配置参数           | 单次请求 |


# 1.Context(运行时上下文)

Runtime Context 是一种依赖注入机制：调用者在运行开始时把用户身份、租户信息、权限或外部服务连接传给 Agent，节点、中间件和工具在执行时再从 Runtime 读取。

它解决的是三个工程问题：
- 避免把用户 ID、数据库连接等信息硬编码进工具。
- 同一个 Agent 可以安全地服务不同用户和租户。
- 工具的业务参数只保留真正需要模型填写的内容。

## 1.1.准备数据

我们先准备一个字典，模拟一个多租户的在线教育项目数据库，其中的租户就是学校，user就是学员：
- 字典key是租户tenant_id，字典value是租户对应的user信息，也是字典。
- user字典的key是user_id

In [2]:
USER_DATABASE = {
    "school_a": {
        "u001": {"name": "虎哥", "course": "LangChain 开发指南"},
        "u002": {"name": "小明", "course": "Java开发入门"},
    },
    "school_b": {
        "u001": {"name": "涛哥", "course": "Python 入门"},
        "u002": {"name": "阿伟", "course": "VibeCoding 实战"},
    }
}

## 1.2.定义ContextSchema

In [3]:
from dataclasses import dataclass

@dataclass(frozen=True)
class UserContext:
    """Agent运行时上下文"""
    user_id: str
    tenant_id: str

## 1.3.在Tools中使用Context

In [4]:
from langchain.tools import tool, ToolRuntime

@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
    """查询当前登录用户的资料。"""

    # 获取contextu
    ctx= runtime.context

    print("ctx : ", ctx)

    # 根据user_id和tenant_id获取用户信息
    profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)

    return f"姓名：{profile['name']}，课程：{profile['course']}"

## 1.4.给Agent添加Context

In [5]:
from langchain.agents import create_agent

# 创建Agent时指定context_schema
agent = create_agent(
    model="deepseek-flash",
    tools=[get_current_user_profile],
    context_schema=UserContext,  # 指定Context类型
    system_prompt="你是课程平台助手。需要用户资料时调用工具。"
)

## 1.5.调用Agent，传递Context信息

In [6]:
from langchain.messages import HumanMessage

# 调用时通过Context传递用户信息
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询我正在学习的课程")]},
    context=UserContext(user_id="u001", tenant_id="school_a")
)

for message in response['messages']:
    message.pretty_print()

ctx :  UserContext(user_id='u001', tenant_id='school_a')
================================ Human Message =================================

Hello, 帮我查询我正在学习的课程
================================== Ai Message ==================================

我来帮您查询，先获取您的资料信息。
Tool Calls:
  get_current_user_profile (call_00_weWWP0bVVHewoiuje7uw8293)
 Call ID: call_00_weWWP0bVVHewoiuje7uw8293
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥，课程：LangChain 开发指南
================================== Ai Message ==================================

您好，虎哥！根据您的资料信息：

- **姓名**：虎哥
- **正在学习的课程**：LangChain 开发指南

您目前正在学习《LangChain 开发指南》这门课程。如果需要了解课程内容、学习进度或其他帮助，随时告诉我～


# 2. State（短期记忆）

State是Agent的短期记忆，存储当前会话的历史消息、任务状态等信息。

我们之前学习短期记忆时使用的是默认的AgentState，其中只包含会话的历史消息，本节开始我们学习如何自定义AgentState，记录除了历史消息以外的其他信息。


## 2.1 自定义State

In [7]:
from langchain.agents import AgentState


# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: int  # 模型调用次数

## 2.2.在工具中访问state

在定义tool的时候，LangChain内置了一个runtime参数，通过runtime我们可以获取Agent的内部信息，包括：
- state
- store
- context

因此，runtime成为了LangChain中tool的限定参数，自定义参数不能叫这个名字。


In [8]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_model_call_count(runtime: ToolRuntime):
    """A tool that count model call """

    # 获取state中的数据
    count = runtime.state.get("model_call_count", 0)
    print("model_call_count:", count)

    # 更新State中的数据
    update = {
        "model_call_count": count + 1, # 更新count值
        "messages": [ # 工具执行结束必须返回一条ToolMessage
            ToolMessage(
                content="Successfully updated agent state",
                tool_call_id=runtime.tool_call_id
            )
        ]
    }

    return Command(update=update)

In [9]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 初始化模型
model = init_chat_model("deepseek-flash", extra_body={"thinking": {"type": "disabled"}})

# 创建Agent
agent = create_agent(
    model,
    tools=[update_model_call_count],
    state_schema=CustomState,  # 告诉Agent使用自定义的state
    checkpointer=InMemorySaver(),
    system_prompt="你是AI助手，你需要在每次回答用户前调用工具记录模型调用次数"
)

In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Hi,我是虎哥")]},
    config
)

for message in response['messages']:
    message.pretty_print()


model_call_count: 0
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_o3LNXXT3glWOuibQVvCC9263)
 Call ID: call_00_o3LNXXT3glWOuibQVvCC9263
  Args:
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated agent state
================================== Ai Message ==================================

你好，虎哥！很高兴见到你 😄

有什么我可以帮你的吗？无论是聊天、解答问题，还是需要帮忙处理点事情，随时开口。


In [11]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi,我是虎哥', additional_kwargs={}, response_metadata={}, id='b3998918-7fe1-42d5-8616-55773e6bd717'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 272, 'total_tokens': 295, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 144}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '06cdeefd-888e-444f-bd4e-acec102960d6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a2de-c678-7671-8678-2f3cb42a99ad-0', tool_calls=[{'name': 'update_model_call_count', 'args': {}, 'id': 'call_00_o3LNXXT3glWOuibQVvCC9263', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="还记得我是谁吗")]},
    config
)

for message in response['messages']:
    message.pretty_print()

model_call_count: 1
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_o3LNXXT3glWOuibQVvCC9263)
 Call ID: call_00_o3LNXXT3glWOuibQVvCC9263
  Args:
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated agent state
================================== Ai Message ==================================

你好，虎哥！很高兴见到你 😄

有什么我可以帮你的吗？无论是聊天、解答问题，还是需要帮忙处理点事情，随时开口。
================================ Human Message =================================

还记得我是谁吗
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_ZpF0FhGJ0BK3hNrF6m0I0404)
 Call ID: call_00_ZpF0FhGJ0BK3hNrF6m0I0404
  Args:
================================= Tool Message =================================
Name: update_model_call_co

In [13]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi,我是虎哥', additional_kwargs={}, response_metadata={}, id='b3998918-7fe1-42d5-8616-55773e6bd717'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 272, 'total_tokens': 295, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 144}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '06cdeefd-888e-444f-bd4e-acec102960d6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a2de-c678-7671-8678-2f3cb42a99ad-0', tool_calls=[{'name': 'update_model_call_count', 'args': {}, 'id': 'call_00_o3LNXXT3glWOuibQVvCC9263', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

# 3. Store（长期记忆）

store是LangChain提供的长期记忆机制，用于在不同会话间共享数据。例如：模型以外的数据、用户偏好等。
LangChain提供了多种Store的实现方式，例如：
- InMemoryStore
- PostgresStore
- RedisStore
- ...

课程中我们以InMemoryStore为例来学习。


## 3.1 初始化Store

In [14]:
from langgraph.store.memory import InMemoryStore

# ==================== 1. 创建Store ====================
memory_store = InMemoryStore()

## 3.2 在tool中访问store

In [15]:
from langchain.tools import ToolRuntime, tool


# 定义方法，根据Context中的tenant_id和user_id拼接得到namespace
def preference_namespace(ctx: UserContext) -> tuple[str, ...]:
    return (
        "heima_platform",
        ctx.tenant_id,
        ctx.user_id
    )
# 定义固定的key
key = "learning_preferences"

@tool
def save_learning_preference(
    preference_name: str,
    preference_value: str,
    runtime: ToolRuntime[UserContext],
) -> str:
    """保存当前用户的长期学习偏好。"""

    # 通过上下文中的tenant_id和user_id来构建namespace
    namespace = preference_namespace(runtime.context)
    print("namespace:", namespace)

    # 通过namespace和key来读取用户偏好, 返回结果item.value是dict格式
    item = runtime.store.get(namespace, key)
    preferences = item.value if item else {}

    # 记录本次要保存的用户偏好
    preferences[preference_name] = preference_value

    # 保存到Store中
    runtime.store.put(namespace, key, preferences)

    return f"已保存偏好：{preference_name}={preference_value}"


@tool
def get_learning_preferences(
    runtime: ToolRuntime[UserContext],
) -> str:
    """读取当前用户的长期学习偏好。"""

    namespace = preference_namespace(runtime.context)
    item = runtime.store.get(namespace, key)

    if item is None:
        return "当前用户还没有保存学习偏好"

    return f"当前学习偏好：{item.value}"

## 3.3.创建Agent，指定Store

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model="deepseek-flash",
    tools=[
        get_current_user_profile,
        save_learning_preference,
        get_learning_preferences,
    ],
    context_schema=UserContext,
    checkpointer=checkpointer,
    store=store,
    system_prompt=(
        "你是课程平台助手。"
        "涉及用户资料时调用资料工具，"
        "涉及长期学习偏好或给用户讲解代码时调用偏好工具。"
    ),
)

In [17]:
context = UserContext(
    user_id="u001",
    tenant_id="school_a",
)

result = agent.invoke(
    {"messages": [HumanMessage("记住：我喜欢先看代码，再看原理")]},
    config={"configurable": {"thread_id": "thread-a"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

namespace: ('heima_platform', 'school_a', 'u001')
================================ Human Message =================================

记住：我喜欢先看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_preference (call_00_BFzToJDf2MJTOMuCYEtS6113)
 Call ID: call_00_BFzToJDf2MJTOMuCYEtS6113
  Args:
    preference_name: 讲解顺序
    preference_value: 先看代码，再看原理
================================= Tool Message =================================
Name: save_learning_preference

已保存偏好：讲解顺序=先看代码，再看原理
================================== Ai Message ==================================

已记住 ✅

**偏好已保存：** 讲解顺序 = 先看代码，再看原理

以后我给你讲解知识点或示例时，会先给出可运行的代码，再展开背后的原理和机制。如果想调整（比如某个主题希望先讲原理），随时告诉我。


In [18]:
result = agent.invoke(
    {"messages": [HumanMessage("langchain如何实现结构化输出？")]},
    config={"configurable": {"thread_id": "thread-b"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

ctx :  UserContext(user_id='u001', tenant_id='school_a')
================================ Human Message =================================

langchain如何实现结构化输出？
================================== Ai Message ==================================
Tool Calls:
  get_learning_preferences (call_00_9Ud2UjD2Sh0BePgoGyz57207)
 Call ID: call_00_9Ud2UjD2Sh0BePgoGyz57207
  Args:
  get_current_user_profile (call_01_R6JfMaIqmlkYu2ENVrTt4091)
 Call ID: call_01_R6JfMaIqmlkYu2ENVrTt4091
  Args:
================================= Tool Message =================================
Name: get_learning_preferences

当前学习偏好：{'讲解顺序': '先看代码，再看原理'}
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥，课程：LangChain 开发指南
================================== Ai Message ==================================

下面按你习惯的顺序来：**先给能直接跑的代码，再讲背后原理和坑**。

## 一、代码：三种最常用的写法

### 1. Pydantic 模型（最推荐）

```python
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, 

# 4. 总结

1. **State（短期记忆）**
   - 通过`runtime.state`参数访问
   - 用于保存会话历史消息、对话状态、计数等临时信息
   - 生命周期是当前会话
   - 存储方式：可以是InMemorySaver、也可以是数据库


2. **Store（长期记忆）**
   - 通过`runtime.store`访问
   - 用于拓展知识、用户偏好等信息
   - 生命周期跨越多个会话
   - 支持InMemoryStore和数据库存储


3. **Context（运行时上下文）**
   - 通过`runtime.context`访问
   - 用于传递用户ID、配置参数等
   - 生命周期是当前会话
   - 存储方式：基于内存


---

**更多资源**:
- [LangChain官方文档 - Runtime](https://docs.langchain.com/oss/python/langchain/runtime)
- [LangChain官方文档 - Long-term Memory](https://docs.langchain.com/oss/python/langchain/long-term-memory)
- [LangGraph Store文档](https://langchain-ai.github.io/langgraph/concepts/store/)